# YOLO + ByteTrack HOTA Full Test/Val Experiment

This notebook evaluates YOLO detection + Ultralytics ByteTrack on the same held-out sequence split used by `notebooks/train_soccer.ipynb`. The training notebook creates only two detection splits: `train` and `val`; here `val` is treated as the full held-out test set.


In [ ]:
from pathlib import Path
import configparser
import json
import random
import shutil

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'outputs').exists():
            return candidate
    return start


def select_device():
    try:
        import torch
        if torch.cuda.is_available() and torch.cuda.device_count() > 0:
            print('Using CUDA:', torch.cuda.get_device_name(0))
            return 'cuda:0'
    except Exception as exc:
        print('CUDA check skipped:', repr(exc))
    print('Using CPU')
    return 'cpu'


PROJECT_ROOT = find_project_root()
RAW_TRAIN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'tracking' / 'train'
FULLFRAME_YOLO_DIR = PROJECT_ROOT / 'data' / 'yolo' / 'fullframe'
YOLO_WEIGHTS = PROJECT_ROOT / 'outputs' / 'detect_player' / 'runs' / 'E1_yolo_fullframe_img960' / 'weights' / 'best.pt'
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments'
RUNS_DIR = EXPERIMENT_DIR / 'runs'

print('Project root:', PROJECT_ROOT)
print('Raw train dir:', RAW_TRAIN_DIR)
print('YOLO dataset:', FULLFRAME_YOLO_DIR)
print('YOLO weights:', YOLO_WEIGHTS)

assert RAW_TRAIN_DIR.exists(), f'Missing raw tracking train dir: {RAW_TRAIN_DIR}'
assert FULLFRAME_YOLO_DIR.exists(), f'Missing YOLO dataset dir: {FULLFRAME_YOLO_DIR}'
assert YOLO_WEIGHTS.exists(), f'Missing YOLO weights: {YOLO_WEIGHTS}'


In [ ]:
# Match the split used in notebooks/train_soccer.ipynb
SPLIT_SEED = 42
VAL_RATIO = 0.2
SPLIT_BY = 'sequence'
EVAL_SPLIT = 'val'  # train notebook has only train/val; val is the held-out test set here.
MAX_FRAMES = None  # Keep None for full-sequence evaluation.
CLEAN_RUN_DIR = True

IMG_SIZE = 960
CONF = 0.05
IOU = 0.60
DEVICE = select_device()

CLASS_NAMES = ['player', 'goalkeeper', 'referee']
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}


def find_sequence_dirs(raw_train_dir: Path):
    return [
        p for p in sorted(raw_train_dir.iterdir())
        if p.is_dir() and (p / 'img1').exists() and (p / 'gt' / 'gt.txt').exists()
    ]


def make_detection_split(raw_train_dir: Path, seed=42, val_ratio=0.2, split_by='sequence'):
    seq_dirs = find_sequence_dirs(raw_train_dir)
    if not seq_dirs:
        raise FileNotFoundError(f'No valid sequence dirs found inside {raw_train_dir}')

    if split_by == 'sequence' and len(seq_dirs) >= 5:
        shuffled = seq_dirs[:]
        random.Random(seed).shuffle(shuffled)
        n_val = max(1, int(len(shuffled) * val_ratio))
        val = sorted(p.name for p in shuffled[:n_val])
        train = sorted(p.name for p in shuffled[n_val:])
    else:
        # The train notebook falls back to frame split only for tiny datasets. SoccerNet train uses sequence split.
        val = []
        train = sorted(p.name for p in seq_dirs)

    return {'train': train, 'val': val}


def yolo_split_sequences(split_name: str):
    img_dir = FULLFRAME_YOLO_DIR / 'images' / split_name
    if not img_dir.exists():
        return []
    return sorted({p.stem.split('_')[0] for p in img_dir.glob('*.jpg')})


split_sequences = make_detection_split(RAW_TRAIN_DIR, SPLIT_SEED, VAL_RATIO, SPLIT_BY)
eval_sequences = split_sequences[EVAL_SPLIT]
yolo_eval_sequences = yolo_split_sequences(EVAL_SPLIT)

print(f'Split seed: {SPLIT_SEED}')
print(f'Val ratio: {VAL_RATIO}')
print(f'Train sequences: {len(split_sequences["train"])}')
print(f'Held-out val/test sequences: {len(eval_sequences)}')
print('Held-out val/test:', ', '.join(eval_sequences))

if yolo_eval_sequences and yolo_eval_sequences != eval_sequences:
    raise AssertionError(
        'Computed split does not match data/yolo/fullframe/images/val.\n'
        f'Computed: {eval_sequences}\n'
        f'YOLO dir: {yolo_eval_sequences}'
    )

run_name = f'{EVAL_SPLIT}_seed{SPLIT_SEED}_full' if MAX_FRAMES is None else f'{EVAL_SPLIT}_seed{SPLIT_SEED}_{MAX_FRAMES}frames'
run_dir = RUNS_DIR / run_name
pred_dir = run_dir / 'preds'
trackeval_dir = run_dir / 'trackeval'
metrics_json = run_dir / 'metrics.json'
metrics_csv = run_dir / 'metrics.csv'
split_json = run_dir / 'split.json'

if CLEAN_RUN_DIR and run_dir.exists():
    shutil.rmtree(run_dir)

pred_dir.mkdir(parents=True, exist_ok=True)
trackeval_dir.mkdir(parents=True, exist_ok=True)

split_json.write_text(json.dumps({
    'seed': SPLIT_SEED,
    'val_ratio': VAL_RATIO,
    'split_by': SPLIT_BY,
    'eval_split': EVAL_SPLIT,
    'max_frames': MAX_FRAMES,
    'train_sequences': split_sequences['train'],
    'eval_sequences': eval_sequences,
}, indent=2), encoding='utf-8')

print('Run dir:', run_dir)
print('Split json:', split_json)


In [ ]:
try:
    import trackeval
    TRACK_EVAL_AVAILABLE = True
    print('TrackEval is available')
except Exception as exc:
    TRACK_EVAL_AVAILABLE = False
    print('TrackEval is not installed.')
    print('Install it with: pip install git+https://github.com/JonathonLuiten/TrackEval.git')
    print('Import error:', repr(exc))


In [ ]:
def read_seqinfo(seq_dir: Path):
    parser = configparser.ConfigParser()
    parser.read(seq_dir / 'seqinfo.ini')
    info = parser['Sequence']
    seq_length = info.getint('seqLength')
    if MAX_FRAMES is not None:
        seq_length = min(seq_length, int(MAX_FRAMES))
    return {
        'name': info.get('name', seq_dir.name),
        'frame_rate': info.getint('frameRate'),
        'seq_length': seq_length,
        'width': info.getint('imWidth'),
        'height': info.getint('imHeight'),
        'im_ext': info.get('imExt', '.jpg'),
    }


def read_track_id_to_class(seq_dir: Path):
    parser = configparser.ConfigParser(strict=False)
    parser.optionxform = str
    parser.read(seq_dir / 'gameinfo.ini')
    mapping = {}
    if 'Sequence' not in parser:
        return mapping
    for key, value in parser['Sequence'].items():
        if not key.startswith('trackletID_'):
            continue
        track_id = int(key.replace('trackletID_', ''))
        desc = value.split(';')[0].strip().lower()
        if 'referee' in desc:
            cls = 'referee'
        elif 'goalkeeper' in desc or 'goalkeepers' in desc:
            cls = 'goalkeeper'
        elif 'player' in desc:
            cls = 'player'
        else:
            cls = None
        if cls in CLASS_TO_ID:
            mapping[track_id] = cls
    return mapping


def load_gt(seq_dir: Path):
    track_to_class = read_track_id_to_class(seq_dir)
    gt = pd.read_csv(seq_dir / 'gt' / 'gt.txt', header=None)
    gt.columns = ['frame', 'track_id', 'x', 'y', 'w', 'h', 'mark', 'c1', 'c2', 'c3'][:gt.shape[1]]
    if MAX_FRAMES is not None:
        gt = gt[gt['frame'] <= int(MAX_FRAMES)]
    gt['class_name'] = gt['track_id'].map(track_to_class)
    gt = gt[gt['class_name'].isin(CLASS_NAMES)].copy()
    gt['class_id'] = gt['class_name'].map(CLASS_TO_ID)
    return gt


seq_dirs = {name: RAW_TRAIN_DIR / name for name in eval_sequences}
seq_infos = {name: read_seqinfo(path) for name, path in seq_dirs.items()}
gt_by_sequence = {name: load_gt(path) for name, path in seq_dirs.items()}

summary_rows = []
for name in eval_sequences:
    counts = gt_by_sequence[name]['class_name'].value_counts().to_dict()
    summary_rows.append({'sequence': name, 'frames': seq_infos[name]['seq_length'], **counts})

sequence_summary_df = pd.DataFrame(summary_rows).fillna(0)
sequence_summary_df


In [ ]:
def run_yolo_bytetrack_sequence(model_path: Path, sequence_name: str, seq_dir: Path, output_txt: Path):
    seq_info = seq_infos[sequence_name]
    image_paths = sorted((seq_dir / 'img1').glob(f'*{seq_info["im_ext"]}'))
    if MAX_FRAMES is not None:
        image_paths = image_paths[:int(MAX_FRAMES)]
    assert image_paths, f'No images found in {seq_dir / img1}'

    # Fresh model per sequence resets ByteTrack state between sequences.
    model = YOLO(str(model_path))
    rows = []

    for frame_idx, image_path in tqdm(enumerate(image_paths, start=1), total=len(image_paths), desc=sequence_name):
        result = model.track(
            str(image_path),
            tracker='bytetrack.yaml',
            persist=True,
            imgsz=IMG_SIZE,
            conf=CONF,
            iou=IOU,
            device=DEVICE,
            verbose=False,
        )[0]

        boxes = result.boxes
        if boxes is None or boxes.id is None:
            continue

        xyxy = boxes.xyxy.cpu().numpy()
        ids = boxes.id.cpu().numpy().astype(int)
        cls_ids = boxes.cls.cpu().numpy().astype(int)
        confs = boxes.conf.cpu().numpy()

        for box, track_id, cls_id, score in zip(xyxy, ids, cls_ids, confs):
            if int(cls_id) not in CLASS_TO_ID.values():
                continue
            x1, y1, x2, y2 = map(float, box)
            rows.append({
                'sequence': sequence_name,
                'frame': frame_idx,
                'track_id': int(track_id),
                'x': x1,
                'y': y1,
                'w': max(0.0, x2 - x1),
                'h': max(0.0, y2 - y1),
                'score': float(score),
                'class_id': int(cls_id),
                'class_name': CLASS_NAMES[int(cls_id)],
            })

    pred = pd.DataFrame(rows)
    output_txt.parent.mkdir(parents=True, exist_ok=True)
    if pred.empty:
        output_txt.write_text('', encoding='utf-8')
        return pred

    pred[['frame', 'track_id', 'x', 'y', 'w', 'h', 'score', 'class_id']].to_csv(
        output_txt, header=False, index=False, float_format='%.3f'
    )
    return pred


pred_dfs = []
for sequence_name in eval_sequences:
    seq_pred_txt = pred_dir / f'{sequence_name}_bytetrack.txt'
    pred = run_yolo_bytetrack_sequence(YOLO_WEIGHTS, sequence_name, seq_dirs[sequence_name], seq_pred_txt)
    pred_dfs.append(pred)
    print(sequence_name, 'pred rows:', len(pred), 'txt:', seq_pred_txt)

pred_df = pd.concat(pred_dfs, ignore_index=True) if pred_dfs else pd.DataFrame()
if pred_df.empty:
    raise RuntimeError('ByteTrack produced no tracked boxes on the held-out split.')

pred_summary = pred_df.groupby(['sequence', 'class_name']).size().unstack(fill_value=0)
pred_summary.to_csv(run_dir / 'prediction_counts.csv')
pred_summary


In [ ]:
def write_seqinfo(source_seq_dir: Path, target_seq_dir: Path, seq_length: int):
    text = (source_seq_dir / 'seqinfo.ini').read_text(encoding='utf-8')
    if MAX_FRAMES is not None:
        lines = []
        for line in text.splitlines():
            if line.startswith('seqLength='):
                lines.append(f'seqLength={seq_length}')
            else:
                lines.append(line)
        text = '\n'.join(lines) + '\n'

    target_seq_dir.mkdir(parents=True, exist_ok=True)
    (target_seq_dir / 'seqinfo.ini').write_text(text, encoding='utf-8')


def write_trackeval_subset(class_name: str):
    class_root = trackeval_dir / class_name
    if class_root.exists():
        shutil.rmtree(class_root)

    tracker_data_dir = class_root / 'trackers' / 'bytetrack' / 'data'
    tracker_data_dir.mkdir(parents=True, exist_ok=True)

    for sequence_name in eval_sequences:
        source_seq_dir = seq_dirs[sequence_name]
        gt_seq_dir = class_root / 'gt' / sequence_name
        (gt_seq_dir / 'gt').mkdir(parents=True, exist_ok=True)
        write_seqinfo(source_seq_dir, gt_seq_dir, seq_infos[sequence_name]['seq_length'])

        gt_part = gt_by_sequence[sequence_name]
        gt_part = gt_part[gt_part['class_name'] == class_name].copy()
        gt_out = pd.DataFrame({
            0: gt_part['frame'].astype(int),
            1: gt_part['track_id'].astype(int),
            2: gt_part['x'],
            3: gt_part['y'],
            4: gt_part['w'],
            5: gt_part['h'],
            6: 1,
            7: 1,  # TrackEval MOTChallenge class id: pedestrian
            8: 1,
        })
        gt_out.to_csv(gt_seq_dir / 'gt' / 'gt.txt', header=False, index=False, float_format='%.3f')

        pred_part = pred_df[(pred_df['sequence'] == sequence_name) & (pred_df['class_name'] == class_name)].copy()
        pred_out = pd.DataFrame({
            0: pred_part['frame'].astype(int),
            1: pred_part['track_id'].astype(int),
            2: pred_part['x'],
            3: pred_part['y'],
            4: pred_part['w'],
            5: pred_part['h'],
            6: pred_part['score'],
            7: -1,
            8: -1,
            9: -1,
        })
        pred_out.to_csv(tracker_data_dir / f'{sequence_name}.txt', header=False, index=False, float_format='%.3f')

    return class_root


subset_roots = {class_name: write_trackeval_subset(class_name) for class_name in CLASS_NAMES}
subset_roots


In [ ]:
if not TRACK_EVAL_AVAILABLE:
    raise ImportError('TrackEval is required. Run: pip install git+https://github.com/JonathonLuiten/TrackEval.git')

# TrackEval still references removed NumPy aliases in some installs.
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'int'):
    np.int = int


def mean_value(value):
    arr = np.asarray(value, dtype=float)
    return float(np.nanmean(arr))


def evaluate_one_class(class_name: str, class_root: Path):
    eval_config = trackeval.Evaluator.get_default_eval_config()
    eval_config.update({
        'PRINT_RESULTS': False,
        'PRINT_ONLY_COMBINED': True,
        'DISPLAY_LESS_PROGRESS': True,
        'OUTPUT_SUMMARY': False,
        'OUTPUT_DETAILED': False,
        'PLOT_CURVES': False,
    })

    dataset_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    dataset_config.update({
        'GT_FOLDER': str(class_root / 'gt'),
        'TRACKERS_FOLDER': str(class_root / 'trackers'),
        'OUTPUT_FOLDER': str(class_root / 'trackeval_output'),
        'TRACKERS_TO_EVAL': ['bytetrack'],
        'CLASSES_TO_EVAL': ['pedestrian'],
        'BENCHMARK': 'MOT17',
        'SPLIT_TO_EVAL': 'train',
        'INPUT_AS_ZIP': False,
        'PRINT_CONFIG': False,
        'DO_PREPROC': False,
        'TRACKER_SUB_FOLDER': 'data',
        'OUTPUT_SUB_FOLDER': '',
        'SEQ_INFO': {name: seq_infos[name]['seq_length'] for name in eval_sequences},
        'SKIP_SPLIT_FOL': True,
    })

    metrics_config = {'METRICS': ['HOTA', 'CLEAR', 'Identity'], 'THRESHOLD': 0.5}
    evaluator = trackeval.Evaluator(eval_config)
    dataset = trackeval.datasets.MotChallenge2DBox(dataset_config)
    metrics = [
        trackeval.metrics.HOTA(metrics_config),
        trackeval.metrics.CLEAR(metrics_config),
        trackeval.metrics.Identity(metrics_config),
    ]

    results, _ = evaluator.evaluate([dataset], metrics)
    combined = results['MotChallenge2DBox']['bytetrack']['COMBINED_SEQ']['pedestrian']
    row = {'class': class_name}
    for metric_name, metric_values in combined.items():
        if isinstance(metric_values, dict):
            for key, value in metric_values.items():
                row[key] = mean_value(value)
    return row


metric_rows = [evaluate_one_class(class_name, subset_roots[class_name]) for class_name in CLASS_NAMES]
metrics_df = pd.DataFrame(metric_rows)
mean_row = metrics_df.select_dtypes(include='number').mean().to_dict()
mean_row['class'] = 'mean'
metrics_df = pd.concat([metrics_df, pd.DataFrame([mean_row])], ignore_index=True)

metrics_df.to_csv(metrics_csv, index=False)
metrics_json.write_text(json.dumps(metric_rows + [mean_row], indent=2), encoding='utf-8')

print('Saved:', metrics_csv)
print('Saved:', metrics_json)
metrics_df


In [ ]:
print('Evaluated split:', EVAL_SPLIT)
print('Sequences:', ', '.join(eval_sequences))
print('Metrics CSV:', metrics_csv)
print('Metrics JSON:', metrics_json)
